# Vehicle Speed Detection using YOLOv8

This notebook implements a vehicle speed detection system using:
- **YOLOv8** for object detection
- **Built-in ByteTrack** for object tracking
- **Distance-time calculation** for speed estimation

In [47]:
# Install required packages
!pip install ultralytics opencv-python pandas -q


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
# Import libraries
import cv2
import time
import os
from ultralytics import YOLO

# Load YOLOv8 model
model = YOLO('yolov8s.pt')
print("Model loaded successfully!")

Model loaded successfully!


In [ ]:
# Configuration
VIDEO_PATH = 'datasets/test1.mp4'  # Input video path
OUTPUT_PATH = 'outputs/test1_baseline_output.avi'  # Output video path
FRAME_WIDTH = 1020
FRAME_HEIGHT = 500

# Speed detection lines (y-coordinates)
LINE_1_Y = 198  # First detection line (red)
LINE_2_Y = 268  # Second detection line (blue)
DISTANCE_METERS = 50  # Real-world distance between lines in meters
OFFSET = 6  # Tolerance for line crossing detection

# Vehicle classes to track (COCO dataset indices)
# 2: car, 3: motorcycle, 5: bus, 7: truck
VEHICLE_CLASSES = [2, 3, 5, 7]

In [50]:
class SpeedDetector:
    def __init__(self, line1_y, line2_y, distance_meters, offset=6):
        self.line1_y = line1_y  # First line (red)
        self.line2_y = line2_y  # Second line (blue)
        self.distance = distance_meters
        self.offset = offset
        
        # Track timestamps when vehicles cross lines
        self.line1_times = {}  # Going down: cross line1 first
        self.line2_times = {}  # Going up: cross line2 first
        
        # Store calculated speeds
        self.speeds = {}
        
        # Count vehicles
        self.count_down = []
        self.count_up = []
    
    def is_crossing_line(self, cy, line_y):
        """Check if center y-coordinate is crossing a line"""
        return line_y - self.offset < cy < line_y + self.offset
    
    def calculate_speed(self, elapsed_time):
        """Calculate speed in km/h from elapsed time"""
        if elapsed_time > 0:
            speed_ms = self.distance / elapsed_time
            speed_kmh = speed_ms * 3.6
            return speed_kmh
        return 0
    
    def update(self, track_id, cx, cy):
        """Update tracking for a vehicle and return speed if calculated"""
        speed = None
        direction = None
        
        # Going DOWN (line1 first, then line2)
        if self.is_crossing_line(cy, self.line1_y):
            if track_id not in self.line1_times:
                self.line1_times[track_id] = time.time()
        
        if track_id in self.line1_times:
            if self.is_crossing_line(cy, self.line2_y):
                if track_id not in self.count_down:
                    elapsed = time.time() - self.line1_times[track_id]
                    speed = self.calculate_speed(elapsed)
                    self.speeds[track_id] = speed
                    self.count_down.append(track_id)
                    direction = 'down'
        
        # Going UP (line2 first, then line1)
        if self.is_crossing_line(cy, self.line2_y):
            if track_id not in self.line2_times:
                self.line2_times[track_id] = time.time()
        
        if track_id in self.line2_times:
            if self.is_crossing_line(cy, self.line1_y):
                if track_id not in self.count_up:
                    elapsed = time.time() - self.line2_times[track_id]
                    speed = self.calculate_speed(elapsed)
                    self.speeds[track_id] = speed
                    self.count_up.append(track_id)
                    direction = 'up'
        
        # Return stored speed if available
        if track_id in self.speeds:
            return self.speeds[track_id], direction
        return None, direction
    
    def get_counts(self):
        return len(self.count_down), len(self.count_up)

print("SpeedDetector class defined!")

SpeedDetector class defined!


In [51]:
def draw_ui(frame, speed_detector):
    """Draw detection lines and counters on frame"""
    # Colors (BGR)
    RED = (0, 0, 255)
    BLUE = (255, 0, 0)
    YELLOW = (0, 255, 255)
    BLACK = (0, 0, 0)
    
    # Draw detection lines
    cv2.line(frame, (100, LINE_1_Y), (900, LINE_1_Y), RED, 2)
    cv2.line(frame, (100, LINE_2_Y), (900, LINE_2_Y), BLUE, 2)
    
    # Line labels
    cv2.putText(frame, 'Line 1', (105, LINE_1_Y - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, RED, 2)
    cv2.putText(frame, 'Line 2', (105, LINE_2_Y - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, BLUE, 2)
    
    # Counter background
    cv2.rectangle(frame, (0, 0), (250, 90), YELLOW, -1)
    
    # Get counts
    count_down, count_up = speed_detector.get_counts()
    
    # Display counts
    cv2.putText(frame, f'Going Down: {count_down}', (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, BLACK, 2)
    cv2.putText(frame, f'Going Up: {count_up}', (10, 60), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, BLACK, 2)
    
    return frame

def draw_vehicle(frame, x1, y1, x2, y2, track_id, speed=None):
    """Draw bounding box and speed for a vehicle"""
    GREEN = (0, 255, 0)
    CYAN = (255, 255, 0)
    WHITE = (255, 255, 255)
    
    # Bounding box
    cv2.rectangle(frame, (x1, y1), (x2, y2), GREEN, 2)
    
    # Track ID
    cv2.putText(frame, f'ID:{track_id}', (x1, y1 - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 2)
    
    # Speed (if calculated)
    if speed is not None:
        cv2.putText(frame, f'{int(speed)} km/h', (x1, y2 + 20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, CYAN, 2)
    
    return frame

print("Drawing functions defined!")

Drawing functions defined!


In [52]:
# Initialize video capture and writer
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise Exception(f"Error: Cannot open video file '{VIDEO_PATH}'")

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video loaded: {total_frames} frames at {fps:.1f} FPS")

# Create output directory
os.makedirs('detected_frames', exist_ok=True)

# Initialize video writer (MJPG codec - reliable on Windows)
fourcc = cv2.VideoWriter_fourcc(*'MJPG')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (FRAME_WIDTH, FRAME_HEIGHT))

# Initialize speed detector
speed_detector = SpeedDetector(LINE_1_Y, LINE_2_Y, DISTANCE_METERS, OFFSET)

print("Initialization complete!")

Video loaded: 1060 frames at 25.0 FPS
Initialization complete!


In [53]:
# Main processing loop
frame_count = 0
start_time = time.time()

print("Processing video...")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_count += 1
    frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
    
    # Run YOLO tracking (using built-in ByteTrack)
    results = model.track(
        frame, 
        persist=True, 
        classes=VEHICLE_CLASSES,
        verbose=False
    )
    
    # Process detections
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)
        
        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            cxw = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            
            # Update speed detector
            speed, direction = speed_detector.update(track_id, cx, cy)
            
            # Draw vehicle with speed
            frame = draw_vehicle(frame, x1, y1, x2, y2, track_id, speed)
    
    # Draw UI elements
    frame = draw_ui(frame, speed_detector)
    
    # Save frame
    cv2.imwrite(f'detected_frames/frame_{frame_count:05d}.jpg', frame)
    out.write(frame)
    
    # Progress update every 100 frames
    if frame_count % 100 == 0:
        elapsed = time.time() - start_time
        fps_processing = frame_count / elapsed
        print(f"Processed {frame_count}/{total_frames} frames ({fps_processing:.1f} FPS)")

# Cleanup
cap.release()
out.release()

# Final statistics
elapsed = time.time() - start_time
count_down, count_up = speed_detector.get_counts()

print(f"\n{'='*50}")
print(f"Processing complete!")
print(f"Total frames: {frame_count}")
print(f"Processing time: {elapsed:.1f} seconds")
print(f"Average FPS: {frame_count/elapsed:.1f}")
print(f"Vehicles going down: {count_down}")
print(f"Vehicles going up: {count_up}")
print(f"Output saved to: {OUTPUT_PATH}")
print(f"{'='*50}")

Processing video...
Processed 100/1060 frames (6.5 FPS)
Processed 200/1060 frames (6.7 FPS)
Processed 300/1060 frames (6.8 FPS)
Processed 400/1060 frames (6.8 FPS)
Processed 500/1060 frames (6.8 FPS)
Processed 600/1060 frames (6.8 FPS)
Processed 700/1060 frames (6.8 FPS)
Processed 800/1060 frames (6.7 FPS)
Processed 900/1060 frames (6.7 FPS)
Processed 1000/1060 frames (6.7 FPS)

Processing complete!
Total frames: 1060
Processing time: 158.0 seconds
Average FPS: 6.7
Vehicles going down: 34
Vehicles going up: 33
Output saved to: de_output.avi


In [54]:
# Display speed statistics for detected vehicles
print("\nSpeed Statistics:")
print("-" * 40)

if speed_detector.speeds:
    speeds = list(speed_detector.speeds.values())
    print(f"Total vehicles with speed: {len(speeds)}")
    print(f"Average speed: {sum(speeds)/len(speeds):.1f} km/h")
    print(f"Max speed: {max(speeds):.1f} km/h")
    print(f"Min speed: {min(speeds):.1f} km/h")
    
    print("\nIndividual speeds:")
    for track_id, speed in speed_detector.speeds.items():
        print(f"  Vehicle {track_id}: {speed:.1f} km/h")
else:
    print("No speeds calculated - check line positions match your video")


Speed Statistics:
----------------------------------------
Total vehicles with speed: 67
Average speed: 43.5 km/h
Max speed: 59.2 km/h
Min speed: 30.7 km/h

Individual speeds:
  Vehicle 2: 33.6 km/h
  Vehicle 19: 42.2 km/h
  Vehicle 29: 41.5 km/h
  Vehicle 6: 49.7 km/h
  Vehicle 63: 50.0 km/h
  Vehicle 40: 49.2 km/h
  Vehicle 83: 43.5 km/h
  Vehicle 43: 52.2 km/h
  Vehicle 36: 43.2 km/h
  Vehicle 23: 33.7 km/h
  Vehicle 101: 32.0 km/h
  Vehicle 119: 49.7 km/h
  Vehicle 112: 40.0 km/h
  Vehicle 128: 37.6 km/h
  Vehicle 106: 40.0 km/h
  Vehicle 154: 47.7 km/h
  Vehicle 166: 38.5 km/h
  Vehicle 190: 30.9 km/h
  Vehicle 211: 46.4 km/h
  Vehicle 129: 40.3 km/h
  Vehicle 206: 39.2 km/h
  Vehicle 230: 46.1 km/h
  Vehicle 9: 33.0 km/h
  Vehicle 262: 34.0 km/h
  Vehicle 266: 42.1 km/h
  Vehicle 265: 30.7 km/h
  Vehicle 208: 37.1 km/h
  Vehicle 282: 43.6 km/h
  Vehicle 286: 40.1 km/h
  Vehicle 258: 33.6 km/h
  Vehicle 278: 52.0 km/h
  Vehicle 317: 35.8 km/h
  Vehicle 323: 55.7 km/h
  Vehicle 34